# Phase 3 — Surrogate Model Training

Train ML surrogates on `output/dataset.csv` (500 SWMM runs) and compare **Random Forest vs XGBoost** on three tasks:

1. `peak_discharge_cfs` — regression
2. *floods?* (`total_flood_vol > 0`) — classification
3. `total_flood_vol` on flooded cases — regression


> **Prerequisite:** run the Phase 2 notebook with `N_RUNS = 500` first so `output/dataset.csv` holds 500 rows.

## 1. Config & load data

In [ ]:
from pathlib import Path
import pandas as pd

# Open this notebook from the project folder or the notebooks folder.
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
assert (PROJECT / "notebooks").is_dir(), "Open from the project or notebooks folder."
DATA = PROJECT / "output" / "dataset.csv"
MODELS_DIR = PROJECT / "output" / "models"
MODELS_DIR.mkdir(exist_ok=True)
SEED = 42

df = pd.read_csv(DATA)
df = df[df["status"] == "ok"].reset_index(drop=True)

FEATURES = ["rain_mult", "imperv_mult", "surf_n", "infil_min", "pipe_n"]
X = df[FEATURES]

n_flood = int((df["total_flood_vol"] > 0).sum())
print(f"Loaded {df.shape[0]} runs x {df.shape[1]} cols | flooded: {n_flood} "
      f"({100*n_flood/len(df):.0f}%)")


## 2. Train/test split

80/20 split, fixed seed, **stratified on the flood class** so both sides contain flooded cases.

In [ ]:
from sklearn.model_selection import train_test_split

y_peak = df["peak_discharge_cfs"]
flooded = (df["total_flood_vol"] > 0).astype(int)

idx_train, idx_test = train_test_split(
    df.index, test_size=0.2, random_state=SEED, stratify=flooded
)
X_train, X_test = X.loc[idx_train], X.loc[idx_test]

print(f"train {len(idx_train)}  test {len(idx_test)}")
print(f"flood rate -> train {flooded.loc[idx_train].mean():.2f}  "
      f"test {flooded.loc[idx_test].mean():.2f}")

## 3. Task 1 — peak discharge (regression)

Random Forest vs XGBoost. Report R² / RMSE / MAE on the held-out test set plus 5-fold CV R² for a stabler estimate.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score

def reg_metrics(y_true, y_pred):
    return {
        "R2":   r2_score(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE":  mean_absolute_error(y_true, y_pred),
    }

peak_models = {
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=SEED),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                            subsample=0.9, random_state=SEED),
}

rows, peak_fitted = [], {}
for name, model in peak_models.items():
    model.fit(X_train, y_peak.loc[idx_train])
    metrics = reg_metrics(y_peak.loc[idx_test], model.predict(X_test))
    metrics["CV_R2"] = cross_val_score(model, X, y_peak, cv=5, scoring="r2").mean()
    rows.append({"task": "peak_discharge", "model": name, **metrics})
    peak_fitted[name] = model

peak_metrics = pd.DataFrame(rows)
best_peak = peak_metrics.sort_values("R2", ascending=False).iloc[0]["model"]
print("Best peak model:", best_peak)
peak_metrics.round(3)

### Predicted vs SWMM + residuals (best peak model)

In [ ]:
import matplotlib.pyplot as plt

model = peak_fitted[best_peak]
actual = y_peak.loc[idx_test]
pred = model.predict(X_test)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(actual, pred, s=18, alpha=0.7)
lims = [actual.min(), actual.max()]
ax[0].plot(lims, lims, "r--")
ax[0].set_xlabel("SWMM peak (CFS)")
ax[0].set_ylabel("surrogate (CFS)")
ax[0].set_title(f"{best_peak}: predicted vs SWMM")
ax[1].scatter(pred, actual - pred, s=18, alpha=0.7)
ax[1].axhline(0, color="r", ls="--")
ax[1].set_xlabel("predicted (CFS)")
ax[1].set_ylabel("residual")
ax[1].set_title("residuals")
plt.tight_layout()
plt.show()

## 4. Task 2a — floods? (classification)

Stage 1 of the two-stage flood model: does this scenario flood at all?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

flood_train, flood_test = flooded.loc[idx_train], flooded.loc[idx_test]

clf_models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                             subsample=0.9, random_state=SEED, eval_metric="logloss"),
}

rows, clf_fitted = [], {}
for name, model in clf_models.items():
    model.fit(X_train, flood_train)
    p = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({
        "task": "floods?", "model": name,
        "accuracy":  accuracy_score(flood_test, p),
        "precision": precision_score(flood_test, p, zero_division=0),
        "recall":    recall_score(flood_test, p, zero_division=0),
        "F1":        f1_score(flood_test, p, zero_division=0),
        "ROC_AUC":   roc_auc_score(flood_test, proba),
    })
    clf_fitted[name] = model

clf_metrics = pd.DataFrame(rows)
best_clf = clf_metrics.sort_values("F1", ascending=False).iloc[0]["model"]
print("Best classifier:", best_clf)
print("\nConfusion matrix [rows=actual, cols=pred] (0=dry, 1=flood):")
print(confusion_matrix(flood_test, clf_fitted[best_clf].predict(X_test)))
clf_metrics.round(3)

## 5. Task 2b — flood volume (regression, flooded cases only)

Stage 2: given that a scenario floods, how much? Trained only on runs with `total_flood_vol > 0`.

In [ ]:
flood_idx = df.index[df["total_flood_vol"] > 0]
volume_train_idx, volume_test_idx = train_test_split(flood_idx, test_size=0.2, random_state=SEED)
flood_volume = df["total_flood_vol"]

vol_models = {
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=SEED),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                            subsample=0.9, random_state=SEED),
}

rows, vol_fitted = [], {}
for name, model in vol_models.items():
    model.fit(X.loc[volume_train_idx], flood_volume.loc[volume_train_idx])
    rows.append({"task": "flood_volume", "model": name,
                 **reg_metrics(flood_volume.loc[volume_test_idx], model.predict(X.loc[volume_test_idx]))})
    vol_fitted[name] = model

vol_metrics = pd.DataFrame(rows)
best_vol = vol_metrics.sort_values("R2", ascending=False).iloc[0]["model"]
print(f"flooded subset: {len(flood_idx)}  (train {len(volume_train_idx)} / test {len(volume_test_idx)})")
print("Best flood-volume model:", best_vol)
vol_metrics.round(4)

## 6. Extreme-case check + runtime speedup

Brief §18: average accuracy can hide extreme-event failure, so score the peak model on the high-discharge tail. Brief §19: quantify the surrogate speedup over SWMM.

In [ ]:
# --- Extreme-case validation: top-20% by SWMM peak in the test set ---
thr = actual.quantile(0.8)
hi = actual >= thr
print(f"Extreme tail (peak >= {thr:.1f} CFS): n={int(hi.sum())}")
print("  full test :", {k: round(v, 3) for k, v in reg_metrics(actual, pred).items()})
print("  extreme   :", {k: round(v, 3) for k, v in reg_metrics(actual[hi], pred[hi]).items()})

# --- Runtime: 10,000 surrogate predictions vs SWMM equivalent ---
import time
Xbig = X.sample(10_000, replace=True, random_state=SEED)
t0 = time.time()
_ = peak_fitted[best_peak].predict(Xbig)
dt = time.time() - t0
swmm_s = 0.037  # sec/run measured in Phase 2
print(f"\nSurrogate: 10,000 predictions in {dt*1000:.1f} ms")
print(f"SWMM equivalent (@ {swmm_s}s/run): {10_000*swmm_s/60:.1f} min")
print(f"Speedup: ~{(10_000*swmm_s)/dt:,.0f}x")

## 7. Feature importance (peak model)

Permutation importance — a preview of the Phase 4 sensitivity analysis.

In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(peak_fitted[best_peak], X_test, actual,
                           n_repeats=20, random_state=SEED)
imp = pd.Series(r.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(color="#3b7dd8")
plt.title(f"Permutation importance — peak discharge ({best_peak})")
plt.xlabel("mean R2 drop when shuffled")
plt.tight_layout()
plt.show()
imp.sort_values(ascending=False).round(3)

## 8. Light hyperparameter tuning (RandomizedSearchCV)

Due-diligence check: does tuning XGBoost beat the defaults? Tuned on the **training set via 5-fold CV only** (test set untouched), 40 random candidates per task. We judge by **CV gain** (the test sets are small, so test deltas can be noise). Peak + classifier are expected near-ceiling; flood volume has the most headroom.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

N_ITER = 40
XGB_SPACE = {
    "n_estimators": randint(200, 700),
    "max_depth": randint(2, 7),
    "learning_rate": uniform(0.01, 0.19),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 8),
    "reg_lambda": uniform(0.0, 3.0),
    "reg_alpha": uniform(0.0, 1.0),
}

tuned_models = {}

def tune_model(task, default, X_train, y_train, X_test, y_test, scoring):
    search = RandomizedSearchCV(
        default, XGB_SPACE, n_iter=N_ITER, cv=5,
        scoring=scoring, random_state=SEED, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    default_cv = cross_val_score(default, X_train, y_train, cv=5, scoring=scoring).mean()
    tuned = search.best_estimator_

    if scoring == "roc_auc":
        metric = "AUC"
        default_test = roc_auc_score(y_test, default.predict_proba(X_test)[:, 1])
        tuned_test = roc_auc_score(y_test, tuned.predict_proba(X_test)[:, 1])
    else:
        metric = "R2"
        default_test = r2_score(y_test, default.predict(X_test))
        tuned_test = r2_score(y_test, tuned.predict(X_test))

    tuned_models[task] = tuned
    return {
        "task": task, "metric": metric,
        "default_CV": default_cv, "tuned_CV": search.best_score_,
        "default_test": default_test, "tuned_test": tuned_test,
    }

# Reuse the default models fitted in the earlier sections.
rows = []
rows.append(tune_model(
    "peak", peak_fitted["XGBoost"],
    X_train, y_peak.loc[idx_train], X_test, y_peak.loc[idx_test], "r2",
))
rows.append(tune_model(
    "flood_volume", vol_fitted["XGBoost"],
    X.loc[volume_train_idx], flood_volume.loc[volume_train_idx],
    X.loc[volume_test_idx], flood_volume.loc[volume_test_idx], "r2",
))
rows.append(tune_model(
    "floods?", clf_fitted["XGBoost"], X_train, flood_train, X_test, flood_test, "roc_auc",
))

tune_compare = pd.DataFrame(rows)
tune_compare["CV_gain"] = tune_compare["tuned_CV"] - tune_compare["default_CV"]
tune_compare["test_gain"] = tune_compare["tuned_test"] - tune_compare["default_test"]
print(tune_compare.round(4).to_string(index=False))


## 9. Save models + metrics

In [ ]:
import joblib

all_metrics = pd.concat([peak_metrics, clf_metrics, vol_metrics], ignore_index=True)
all_metrics.to_csv(PROJECT / "output" / "phase3_metrics.csv", index=False)

joblib.dump(peak_fitted[best_peak], MODELS_DIR / "peak_regressor.joblib")
joblib.dump(clf_fitted[best_clf],   MODELS_DIR / "flood_classifier.joblib")
joblib.dump(vol_fitted[best_vol],   MODELS_DIR / "flood_volume_regressor.joblib")

print("Saved -> output/phase3_metrics.csv and output/models/*.joblib")
print(f"Best models: peak={best_peak}, floods?={best_clf}, volume={best_vol}")
all_metrics.round(3)